# Limpieza, Feature Engineering y Entrenamiento
**Workshop 3 · Parte A · Pasos 2, 3 y 4.** Este notebook construye el dataset unificado, justifica las decisiones de limpieza, selecciona features, entrena un modelo de regresión simple y serializa **preprocesamiento + modelo** en `models/model.pkl`.

> Ejecutar desde `notebooks/`. Genera: `data/processed/happiness_unified.csv`, `data/streaming/stream_events.csv`, `models/model.pkl`, `models/features.json`, `models/metrics.json`.

In [ ]:
import json, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.width", 200)
RAW, PROCESSED, STREAMING, MODELS = "../data/raw", "../data/processed", "../data/streaming", "../models"
YEARS = [2015, 2016, 2017, 2018, 2019]
SEED = 42

## Paso 2 — Limpieza y armonización

Mapa de renombrado hacia el esquema unificado (ver EDA, sección 8).

In [ ]:
RENAME = {
 "Country":"country", "Country or region":"country",
 "Happiness Score":"happiness_score", "Happiness.Score":"happiness_score", "Score":"happiness_score",
 "Economy (GDP per Capita)":"gdp", "Economy..GDP.per.Capita.":"gdp", "GDP per capita":"gdp",
 "Family":"family", "Social support":"family",
 "Health (Life Expectancy)":"health", "Health..Life.Expectancy.":"health", "Healthy life expectancy":"health",
 "Freedom":"freedom", "Freedom to make life choices":"freedom",
 "Generosity":"generosity",
 "Trust (Government Corruption)":"corruption", "Trust..Government.Corruption.":"corruption", "Perceptions of corruption":"corruption",
}
UNIFIED_COLS = ["country", "year", "happiness_score", "gdp", "family", "health", "freedom", "generosity", "corruption"]
FEATURES = ["gdp", "family", "health", "freedom", "generosity", "corruption"]
TARGET = "happiness_score"

# Nombres de país distintos para la misma entidad
COUNTRY_ALIASES = {
 "Hong Kong S.A.R., China": "Hong Kong",
 "Taiwan Province of China": "Taiwan",
 "Macedonia": "North Macedonia",
 "Northern Cyprus": "North Cyprus",
 "Somaliland Region": "Somaliland region",
 "Trinidad & Tobago": "Trinidad and Tobago",
}

frames = []
for y in YEARS:
    df = pd.read_csv(f"{RAW}/{y}.csv").rename(columns=RENAME)
    df["year"] = y
    frames.append(df[UNIFIED_COLS])
unified = pd.concat(frames, ignore_index=True)
print("Filas antes de limpiar:", len(unified))

In [ ]:
# 1) Texto: quitar espacios y unificar nombres de país
unified["country"] = unified["country"].str.strip().replace(COUNTRY_ALIASES)

# 2) Tipos: year entero; features y target float
unified["year"] = unified["year"].astype(int)
for c in FEATURES + [TARGET]:
    unified[c] = pd.to_numeric(unified[c], errors="coerce").astype(float)

# 3) Faltantes: eliminar filas incompletas (solo UAE-2018 no tiene 'corruption')
missing_rows = unified[unified[FEATURES + [TARGET]].isna().any(axis=1)]
print("Filas con faltantes eliminadas:\n", missing_rows)
unified = unified.dropna(subset=FEATURES + [TARGET])

# 4) Duplicados: ninguna fila completa ni (country, year) repetido
print("Duplicados (country, year):", unified.duplicated(["country", "year"]).sum())
unified = unified.drop_duplicates(["country", "year"])

# 5) Rangos imposibles (valores negativos o score fuera de 0-10)
bad = unified[(unified[FEATURES] < 0).any(axis=1) | ~unified[TARGET].between(0, 10)]
print("Filas con valores imposibles:", len(bad))
unified = unified.drop(bad.index).sort_values(["year", "country"]).reset_index(drop=True)

print("Filas finales:", len(unified)); unified.head()

### Justificación de las decisiones de limpieza
| Decisión | Justificación |
|---|---|
| Renombrar a `snake_case` con un mapa explícito | Un único nombre por concepto; reproducible y auditable. Coincide con los campos del evento Kafka. |
| Unificar nombres de país (6 alias) | Evita tratar `Taiwan` y `Taiwan Province of China` como países distintos; el país es la **llave del evento** y de `dim_country`. |
| `year` tomado del nombre del archivo | Ninguno de los CSV trae el año como columna. |
| Eliminar la fila con `corruption` faltante (UAE-2018) | 1 fila de ~780. Imputar inventaría un dato; eliminarla no afecta el entrenamiento. |
| **No** eliminar outliers de `corruption`/`generosity` | Son valores reales y plausibles; eliminarlos sesgaría el modelo contra países reales. |
| Descartar `Region`, `Rank`, errores estándar, intervalos, whiskers, `Dystopia Residual` | No existen en los 5 años y/o causan *target leakage* (el score es suma de los factores + dystopia residual; el rank es función del score). |

In [ ]:
unified.to_csv(f"{PROCESSED}/happiness_unified.csv", index=False)
unified.groupby("year").size().rename("filas por año").to_frame().T

## Paso 3 — Feature Engineering

In [ ]:
unified[FEATURES + [TARGET]].describe().T

In [ ]:
corr = unified[FEATURES + [TARGET]].corr()
print(corr[TARGET].sort_values(ascending=False).round(3))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); plt.title("Matriz de correlación"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["gdp", "family", "health"]):
    ax.scatter(unified[c], unified[TARGET], alpha=.4, s=12)
    ax.set_xlabel(c); ax.set_ylabel("happiness_score"); ax.set_title(f"{c} vs score")
plt.tight_layout(); plt.show()

### Selección de features y justificación
- **Features:** `gdp`, `family`, `health`, `freedom`, `generosity`, `corruption` (las 6 variables presentes en los 5 años y definidas en el contrato del evento).
- `gdp`, `family` y `health` tienen la correlación más alta con el score; `freedom` es intermedia; `generosity` y `corruption` aportan menos, pero se conservan porque el contrato las exige y el modelo debe recibir el mismo vector de entrada que llega por Kafka.
- **Sin target leakage:** `happiness_score` es solo el *target* (en el evento es `actual_happiness_score`, usado únicamente para evaluar). Se excluyen `Dystopia Residual` y `Rank`.
- **`country` y `year` no son features:** `country` es una etiqueta (llave del evento) y usarla exigiría codificación categórica con alta cardinalidad; `year` solo identifica la observación histórica. No hay datos categóricos en el vector de entrada.
- **Escalado:** `StandardScaler` dentro de un `Pipeline` de scikit-learn; se aprende solo con el set de entrenamiento y se serializa junto con el modelo, de modo que el consumer aplica exactamente el mismo preprocesamiento.

## Paso 4 — Entrenamiento y evaluación (70% / 30%)

In [ ]:
X, y = unified[FEATURES], unified[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED)
print("train:", X_train.shape, "| test:", X_test.shape)

candidates = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "RandomForestRegressor": Pipeline([("scaler", StandardScaler()),
                                       ("model", RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=SEED, n_jobs=-1))]),
}
results = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {"MAE": mean_absolute_error(y_test, pred),
                     "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
                     "R2": r2_score(y_test, pred)}
metrics = pd.DataFrame(results).T.round(4); metrics

In [ ]:
best_name = metrics["RMSE"].idxmin()
best_pipe = candidates[best_name]
print("Modelo seleccionado:", best_name)

pred = best_pipe.predict(X_test)
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=.5, s=14); lim = [y.min(), y.max()]
plt.plot(lim, lim, "r--"); plt.xlabel("Real"); plt.ylabel("Predicho"); plt.title(f"{best_name}: real vs predicho")
plt.tight_layout(); plt.show()

### Resultado
El objetivo del workshop **no es maximizar la precisión** sino integrar el pipeline; un modelo simple con R² alto es suficiente. Se elige el de menor RMSE en el set de prueba.

In [ ]:
joblib.dump(best_pipe, f"{MODELS}/model.pkl")               # preprocesamiento + modelo
json.dump({"features": FEATURES, "target": TARGET, "model": best_name}, open(f"{MODELS}/features.json", "w"), indent=2)
json.dump({"model": best_name, **results[best_name], "all_models": results, "n_train": len(X_train), "n_test": len(X_test)},
          open(f"{MODELS}/metrics.json", "w"), indent=2)

# Recarga de verificación: misma predicción que antes de guardar
loaded = joblib.load(f"{MODELS}/model.pkl")
assert np.allclose(loaded.predict(X_test), pred)
print("model.pkl guardado y verificado")

## Datos para el streaming
El producer lee `data/streaming/stream_events.csv`: el **30 % de prueba** (filas que el modelo no vio al entrenar), así el dashboard mide error de predicción real y no memorización. Se ordena por año para que la tendencia en el tiempo sea legible.

In [ ]:
stream = unified.loc[X_test.index].sort_values(["year", "country"]).reset_index(drop=True)
stream.to_csv(f"{STREAMING}/stream_events.csv", index=False)
print("Eventos para streaming:", len(stream)); stream.head()